In [8]:
# !pip install optuna lightgbm pandas scikit-learn matplotlib

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load data (replace with your dataset path)
X_train = pd.read_parquet('temp/X_resampled.parquet')
X_val = pd.read_parquet('temp/X_val.parquet')


y_train = X_train['TARGET']
y_val = X_val['TARGET']

X_train = X_train.drop(columns=['TARGET'])
X_val = X_val.drop(columns=['TARGET'])

In [10]:
import pandas as pd
df = pd.read_csv('temp/feature_importance.csv')
df.to_excel('temp/feature_importance.xlsx', index=False)

In [11]:
X_train

,NAME_TYPE_SUITE_Children,NAME_TYPE_SUITE_Family,NAME_TYPE_SUITE_Group of people,NAME_TYPE_SUITE_Other,"NAME_TYPE_SUITE_Spouse, partner",NAME_TYPE_SUITE_Unaccompanied,NAME_INCOME_TYPE_Commercial associate,NAME_INCOME_TYPE_Others,NAME_INCOME_TYPE_Pensioner,NAME_INCOME_TYPE_State servant,...,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,EXT_SOURCE_MISSING_VALUES,HAS_DOCUMENT,DOCUMENT_COUNT,RELIABILITY_IN_CUSTOMER_CITY,MISSING_GRADINGS,WEAK_FEATURE,WEAK_FEATURE_2
0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.333333,1.0,0.333333,0.000000,1.000000,0.134,0.078
1,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.333333,1.0,0.333333,0.000000,0.333333,0.130,0.080
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,1.0,0.333333,0.000000,0.000000,0.248,0.138
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.333333,1.0,0.333333,0.000000,0.000000,0.034,0.054
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.666667,1.0,0.333333,0.000000,1.000000,0.106,0.040
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
292078,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.333333,1.0,0.333333,0.333333,1.000000,0.196,0.056
292079,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.000000,1.0,0.333333,0.000000,0.000000,0.006,0.048
292080,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.333333,1.0,0.333333,0.000000,1.000000,0.182,0.072
292081,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,0.333333,1.0,0.333333,0.500000,0.000000,0.030,0.052


In [12]:
X_train.columns = X_train.columns.str.replace(r'[^\w]', '_', regex=True)
X_val.columns = X_val.columns.str.replace(r'[^\w]', '_', regex=True)

In [13]:
import optuna
import lightgbm as lgb
import numpy as np

def objective(trial):
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 1e-1),
        'num_leaves': trial.suggest_int('num_leaves', 10, 250),
        'max_depth': trial.suggest_int('max_depth', 8, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 200),
        'feature_fraction': trial.suggest_uniform('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_loguniform('lambda_l1', 1e-3, 10.0),
        'lambda_l2': trial.suggest_loguniform('lambda_l2', 1e-3, 10.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),
        'n_jobs': -1
    }
    
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    model = lgb.train(
        param,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=1000,
    )
    
    preds = model.predict(X_val)
    auc = roc_auc_score(y_val, preds)
    return auc


In [7]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=300)

# Print the best parameters
print("Best parameters:", study.best_params)
print("Best AUC:", study.best_value)

[I 2024-11-29 19:37:46,435] A new study created in memory with name: no-name-fedf741a-e978-4638-a225-04c19741d3fb
/tmp/ipykernel_85979/3767130388.py:10: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 1e-1),
/tmp/ipykernel_85979/3767130388.py:14: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'feature_fraction': trial.suggest_uniform('feature_fraction', 0.5, 1.0),
/tmp/ipykernel_85979/3767130388.py:15: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'bagging_fraction': trial.suggest_unifo

[LightGBM] [Warning] min_data_in_leaf is set=178, min_child_samples=12 will be ignored. Current value: min_data_in_leaf=178
[LightGBM] [Warning] min_data_in_leaf is set=178, min_child_samples=12 will be ignored. Current value: min_data_in_leaf=178
[LightGBM] [Info] Number of positive: 88563, number of negative: 203520
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.363060 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 147352
[LightGBM] [Info] Number of data points in the train set: 292083, number of used features: 930
[LightGBM] [Warning] min_data_in_leaf is set=178, min_child_samples=12 will be ignored. Current value: min_data_in_leaf=178
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.303212 -> initscore=-0.832050
[LightGBM] [Info] Start training from score -0.832050
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

[I 2024-11-29 19:39:42,460] Trial 0 finished with value: 0.7731351626319185 and parameters: {'learning_rate': 0.07916829632133451, 'num_leaves': 232, 'max_depth': 11, 'min_data_in_leaf': 178, 'feature_fraction': 0.5643123713843397, 'bagging_fraction': 0.8340630347458113, 'bagging_freq': 6, 'lambda_l1': 0.005211807269130363, 'lambda_l2': 0.004055265901324592, 'min_child_samples': 12}. Best is trial 0 with value: 0.7731351626319185.
/tmp/ipykernel_85979/3767130388.py:10: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 1e-1),
/tmp/ipykernel_85979/3767130388.py:14: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'feature_fraction':

[LightGBM] [Warning] min_data_in_leaf is set=88, min_child_samples=61 will be ignored. Current value: min_data_in_leaf=88
[LightGBM] [Warning] min_data_in_leaf is set=88, min_child_samples=61 will be ignored. Current value: min_data_in_leaf=88
[LightGBM] [Info] Number of positive: 88563, number of negative: 203520
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.400490 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 147352
[LightGBM] [Info] Number of data points in the train set: 292083, number of used features: 930
[LightGBM] [Warning] min_data_in_leaf is set=88, min_child_samples=61 will be ignored. Current value: min_data_in_leaf=88
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.303212 -> initscore=-0.832050
[LightGBM] [Info] Start training from score -0.832050


[I 2024-11-29 19:42:15,157] Trial 1 finished with value: 0.7731883940622464 and parameters: {'learning_rate': 0.0025288047387416208, 'num_leaves': 102, 'max_depth': 14, 'min_data_in_leaf': 88, 'feature_fraction': 0.7185009831971099, 'bagging_fraction': 0.5165363588560319, 'bagging_freq': 6, 'lambda_l1': 0.036937904154148216, 'lambda_l2': 0.4438333166324927, 'min_child_samples': 61}. Best is trial 1 with value: 0.7731883940622464.
/tmp/ipykernel_85979/3767130388.py:10: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 1e-1),
/tmp/ipykernel_85979/3767130388.py:14: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'feature_fraction': 

...

[LightGBM] [Warning] min_data_in_leaf is set=60, min_child_samples=181 will be ignored. Current value: min_data_in_leaf=60
[LightGBM] [Warning] min_data_in_leaf is set=60, min_child_samples=181 will be ignored. Current value: min_data_in_leaf=60
[LightGBM] [Info] Number of positive: 88563, number of negative: 203520
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.364614 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 147354
[LightGBM] [Info] Number of data points in the train set: 292083, number of used features: 931
[LightGBM] [Warning] min_data_in_leaf is set=60, min_child_samples=181 will be ignored. Current value: min_data_in_leaf=60
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.303212 -> initscore=-0.832050
[LightGBM] [Info] Start training from score -0.832050


[I 2024-11-29 22:37:35,247] Trial 60 finished with value: 0.7842665598686533 and parameters: {'learning_rate': 0.004484068806106533, 'num_leaves': 185, 'max_depth': 13, 'min_data_in_leaf': 60, 'feature_fraction': 0.5016506589706795, 'bagging_fraction': 0.9981160613268707, 'bagging_freq': 2, 'lambda_l1': 0.016963184292713242, 'lambda_l2': 0.004161079249416208, 'min_child_samples': 181}. Best is trial 23 with value: 0.7922514078856903.
/tmp/ipykernel_85979/3767130388.py:10: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 1e-1),
/tmp/ipykernel_85979/3767130388.py:14: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'feature_fractio

[LightGBM] [Warning] min_data_in_leaf is set=74, min_child_samples=174 will be ignored. Current value: min_data_in_leaf=74
[LightGBM] [Warning] min_data_in_leaf is set=74, min_child_samples=174 will be ignored. Current value: min_data_in_leaf=74
[LightGBM] [Info] Number of positive: 88563, number of negative: 203520
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.329517 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 147352
[LightGBM] [Info] Number of data points in the train set: 292083, number of used features: 930
[LightGBM] [Warning] min_data_in_leaf is set=74, min_child_samples=174 will be ignored. Current value: min_data_in_leaf=74
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.303212 -> initscore=-0.832050
[LightGBM] [Info] Start training from score -0.832050


[I 2024-11-29 22:40:41,191] Trial 61 finished with value: 0.7912067327237648 and parameters: {'learning_rate': 0.0111899757665048, 'num_leaves': 199, 'max_depth': 14, 'min_data_in_leaf': 74, 'feature_fraction': 0.513047936500692, 'bagging_fraction': 0.9776781811860922, 'bagging_freq': 4, 'lambda_l1': 6.2333775538732805, 'lambda_l2': 0.3292121361041549, 'min_child_samples': 174}. Best is trial 23 with value: 0.7922514078856903.
/tmp/ipykernel_85979/3767130388.py:10: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 1e-1),
/tmp/ipykernel_85979/3767130388.py:14: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'feature_fraction': tri

[LightGBM] [Warning] min_data_in_leaf is set=68, min_child_samples=167 will be ignored. Current value: min_data_in_leaf=68
[LightGBM] [Warning] min_data_in_leaf is set=68, min_child_samples=167 will be ignored. Current value: min_data_in_leaf=68
[LightGBM] [Info] Number of positive: 88563, number of negative: 203520
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.317240 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 147354
[LightGBM] [Info] Number of data points in the train set: 292083, number of used features: 931
[LightGBM] [Warning] min_data_in_leaf is set=68, min_child_samples=167 will be ignored. Current value: min_data_in_leaf=68
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.303212 -> initscore=-0.832050
[LightGBM] [Info] Start training from score -0.832050


In [ ]:
{'learning_rate': 0.009407721399062104, 'num_leaves': 198, 'max_depth': 13, 'min_data_in_leaf': 99, 'feature_fraction': 0.5038532461662355, 'bagging_fraction': 0.9912874057611964, 'bagging_freq': 4, 'lambda_l1': 8.040904175809203, 'lambda_l2': 0.1439055371441105, 'min_child_samples': 196}

In [ ]:
best_params = study.best_params
best_params['objective'] = 'binary'
best_params['metric'] = 'auc'

In [ ]:
param = {'learning_rate': 0.07489690004483521, 'num_leaves': 102, 'max_depth': 15, 'min_data_in_leaf': 189, 'feature_fraction': 0.5372551032070825, 'bagging_fraction': 0.810500564688185, 'bagging_freq': 1, 'lambda_l1': 6.0455562622501855, 'lambda_l2': 0.293317330622806, 'min_child_samples': 82}



In [ ]:
best_params

In [ ]:
best_params = param
best_params['objective'] = 'binary'
best_params['metric'] = 'auc'

print("Training the final model with the best parameters")
print(best_params)

final_model = lgb.train(
    best_params,
    lgb.Dataset(X_train, label=y_train),
    num_boost_round=1000
)

In [ ]:
y_pred = final_model.predict(X_val)

In [ ]:
roc_auc_score(y_val,y_pred)

In [ ]:
# Get feature importance and feature names
importance = final_model.feature_importance(importance_type='gain')  # 'gain' measures the contribution
importance_split = final_model.feature_importance(importance_type='split')
feature_names = X_train.columns

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance,
    'Num_Split': importance_split
}).sort_values(by='Importance', ascending=False).sort_values(by='Num_Split', ascending=False)

# Display the most important feature
best_feature = importance_df.iloc[0]
print(f"The most important feature is: {best_feature['Feature']} with an importance score of {best_feature['Importance']}")

# Optional: Display the top 5 features
print("\nTop 5 Features:")
print(importance_df.head())


In [ ]:
importance_df.to_csv('temp/feature_importance.csv', index=False)

In [ ]:
# Get feature importance and feature names
importance = final_model.feature_importance(importance_type='gain')  # 'gain' measures the contribution
importance_split = final_model.feature_importance(importance_type='split')
feature_names = X_train.columns

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance,
    'Num_Split': importance_split
}).sort_values(by='Importance', ascending=False).sort_values(by='Num_Split', ascending=False)

# Display the most important feature
best_feature = importance_df.iloc[0]
print(f"The most important feature is: {best_feature['Feature']} with an importance score of {best_feature['Importance']}")

# Optional: Display the top 5 features
print("\nTop 5 Features:")
print(importance_df.head())

In [ ]:
importance_df.to_excel('temp/feature_importance.xlsx', index=False)